<a href="https://colab.research.google.com/github/srikanthprabala/TorchCode/blob/master/templates/11_sliding_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00


In [7]:
import torch
import math
import torch.nn.functional as F

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE


# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class SlidingWindowAttention(nn.Module):
#     def __init__(self, d_model, num_heads, window_size):
#         super().__init__()
#         assert d_model % num_heads == 0

#         self.d_model = d_model
#         self.num_heads = num_heads
#         self.d_k = d_model // num_heads
#         self.window_size = window_size  # w

#         # projections
#         self.W_q = nn.Linear(d_model, d_model)
#         self.W_k = nn.Linear(d_model, d_model)
#         self.W_v = nn.Linear(d_model, d_model)
#         self.W_o = nn.Linear(d_model, d_model)

#     def forward(self, x):
#         """
#         x: [B, S, D]
#         """
#         B, S, D = x.shape
#         w = self.window_size

#         # 1. Project
#         Q = self.W_q(x)  # [B, S, D]
#         K = self.W_k(x)
#         V = self.W_v(x)

#         # 2. Reshape into heads
#         Q = Q.view(B, S, self.num_heads, self.d_k).transpose(1, 2)  # [B, h, S, d_k]
#         K = K.view(B, S, self.num_heads, self.d_k).transpose(1, 2)
#         V = V.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

#         # 3. Pad K and V so window works at edges
#         K = F.pad(K, (0, 0, w, w))  # pad sequence dim
#         V = F.pad(V, (0, 0, w, w))

#         # After padding:
#         # K: [B, h, S + 2w, d_k]

#         # 4. Extract sliding windows using unfold
#         # We want for each position i → window of size (2w+1)

#         K_windows = K.unfold(dimension=2, size=2*w+1, step=1)
#         V_windows = V.unfold(dimension=2, size=2*w+1, step=1)

#         # K_windows: [B, h, S, 2w+1, d_k]
#         # V_windows: [B, h, S, 2w+1, d_k]

#         # 5. Compute attention
#         Q = Q.unsqueeze(-2)  # [B, h, S, 1, d_k]

#         scores = torch.matmul(Q, K_windows.transpose(-2, -1))
#         # scores: [B, h, S, 1, 2w+1]

#         scores = scores / (self.d_k ** 0.5)

#         attn = torch.softmax(scores, dim=-1)
#         # [B, h, S, 1, 2w+1]

#         # 6. Weighted sum
#         out = torch.matmul(attn, V_windows)
#         # [B, h, S, 1, d_k]

#         out = out.squeeze(-2)  # [B, h, S, d_k]

#         # 7. Merge heads
#         out = out.transpose(1, 2).contiguous().view(B, S, D)

#         return self.W_o(out)


def sliding_window_attention(Q, K, V, window_size):
    """
    Q, K, V: [B, S, d_k]
    Returns: [B, S, d_k]
    """
    B, S, d_k = Q.shape
    w = window_size

    # 1. Pad K and V on sequence dimension
    K = F.pad(K, (0, 0, w, w))  # [B, S + 2w, d_k]
    V = F.pad(V, (0, 0, w, w))

    # # 2. Create sliding windows
    # # → [B, S, 2w+1, d_k]
    # K_windows = K.unfold(dimension=1, size=2*w+1, step=1)
    # V_windows = V.unfold(dimension=1, size=2*w+1, step=1)

    # 2. Unfold + FIX PERMUTE
    K_windows = K.unfold(1, 2*w+1, 1).permute(0, 1, 3, 2)  # [B, S, 2w+1, d_k]
    V_windows = V.unfold(1, 2*w+1, 1).permute(0, 1, 3, 2)

    # 3. Compute attention locally
    # Q: [B, S, d_k] → [B, S, 1, d_k]
    Q = Q.unsqueeze(2)

    # scores: [B, S, 1, 2w+1]
    scores = torch.matmul(Q, K_windows.transpose(-2, -1)) / math.sqrt(d_k)

    # 4. Softmax over local window
    weights = torch.softmax(scores, dim=-1)

    # 5. Weighted sum
    # → [B, S, 1, d_k]
    out = torch.matmul(weights, V_windows)

    # → [B, S, d_k]
    return out.squeeze(2)

In [11]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [12]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (0.7ms)
  ✅ [2/5] window_size=0 — only sees itself (0.6ms)
  ❌ [3/5] Large window equals full attention
     Large window should equal full attention
  ✅ [4/5] Distant tokens don't affect output (4.5ms)
  ✅ [5/5] Gradient flow (33.3ms)
──────────────────────────────────────────────────
  📊 4/5 tests passed.
  Keep going! Use hint("sliding_window") if you're stuck.

